# 11 - Clássicos, Mandante por Clube e Efeito Novo Técnico

- "Efeito Fla-Flu": clássicos têm mais gols e cartões?
- Time grande sofre mais pressão em casa? % vitórias casa vs fora
- Troca de técnico melhora ou piora nos 5 jogos seguintes?

In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

df = pd.read_csv('../data/raw/campeonato-brasileiro-full.csv')
df['data'] = pd.to_datetime(df['data'], format='%d/%m/%Y')
df['ano'] = df['data'].dt.year
df = df.dropna(subset=['rodata'])
df['rodata'] = df['rodata'].astype(int)
df = df[df['ano'] >= 2003].copy()
df['total_gols'] = df['mandante_Placar'] + df['visitante_Placar']

df_cartoes = pd.read_csv('../data/raw/campeonato-brasileiro-cartoes.csv')

# Cartões por partida
cartoes_partida = df_cartoes.groupby('partida_id').size().reset_index(name='total_cartoes')
df = df.merge(cartoes_partida.rename(columns={'partida_id': 'ID'}), on='ID', how='left')
df['total_cartoes'] = df['total_cartoes'].fillna(0)

print(f'Partidas: {len(df)}')

Partidas: 9225


In [2]:
# EFEITO CLÁSSICO
# Definir clássicos (mesmo estado, times grandes)
CLASSICOS = {
    'Fla-Flu': ('Flamengo', 'Fluminense'),
    'Fla x Vasco': ('Flamengo', 'Vasco'),
    'Fla x Botafogo': ('Flamengo', 'Botafogo'),
    'Flu x Vasco': ('Fluminense', 'Vasco'),
    'Corinthians x Palmeiras': ('Corinthians', 'Palmeiras'),
    'Corinthians x S\u00e3o Paulo': ('Corinthians', 'S\u00e3o Paulo'),
    'Palmeiras x S\u00e3o Paulo': ('Palmeiras', 'S\u00e3o Paulo'),
    'Corinthians x Santos': ('Corinthians', 'Santos'),
    'Gre-Nal': ('Gr\u00eamio', 'Internacional'),
    'Atl\u00e9tico x Cruzeiro': ('Atl\u00e9tico-MG', 'Cruzeiro'),
}

def eh_classico(row):
    for nome, (t1, t2) in CLASSICOS.items():
        if (row['mandante'] == t1 and row['visitante'] == t2) or \
           (row['mandante'] == t2 and row['visitante'] == t1):
            return nome
    return None

df['classico'] = df.apply(eh_classico, axis=1)
df['eh_classico'] = df['classico'].notna()

# Comparar médias
print('=== Cl\u00e1ssicos vs Jogos Normais ===')
for metrica in ['total_gols', 'total_cartoes']:
    classicos = df[df['eh_classico']][metrica].mean()
    normais = df[~df['eh_classico']][metrica].mean()
    diff = ((classicos - normais) / normais) * 100
    print(f'\n{metrica}:')
    print(f'  Cl\u00e1ssicos: {classicos:.2f}')
    print(f'  Normais: {normais:.2f}')
    print(f'  Diferen\u00e7a: {diff:+.1f}%')

=== Clássicos vs Jogos Normais ===

total_gols:
  Clássicos: 2.32
  Normais: 2.57
  Diferença: -9.4%

total_cartoes:
  Clássicos: 2.70
  Normais: 2.26
  Diferença: +19.4%


In [3]:
# Gráfico: clássicos vs normais
classico_stats = df.groupby('classico').agg(
    gols=('total_gols', 'mean'),
    cartoes=('total_cartoes', 'mean'),
    jogos=('ID', 'count')
).dropna().sort_values('gols', ascending=True)

# Adicionar média geral
media_geral_gols = df['total_gols'].mean()
media_geral_cartoes = df['total_cartoes'].mean()

fig = make_subplots(rows=1, cols=2, subplot_titles=['M\u00e9dia de Gols', 'M\u00e9dia de Cart\u00f5es'])

fig.add_trace(go.Bar(
    y=classico_stats.index,
    x=classico_stats['gols'],
    orientation='h',
    marker_color='#e74c3c',
    name='Gols',
    text=classico_stats['gols'].round(2),
    textposition='outside',
    hovertemplate='%{y}: %{x:.2f} gols/jogo<extra></extra>'
), row=1, col=1)

fig.add_vline(x=media_geral_gols, line_dash='dash', line_color='gray',
              annotation_text=f'M\u00e9dia: {media_geral_gols:.2f}', row=1, col=1)

fig.add_trace(go.Bar(
    y=classico_stats.index,
    x=classico_stats['cartoes'],
    orientation='h',
    marker_color='#f39c12',
    name='Cart\u00f5es',
    text=classico_stats['cartoes'].round(1),
    textposition='outside',
    hovertemplate='%{y}: %{x:.1f} cart\u00f5es/jogo<extra></extra>'
), row=1, col=2)

fig.add_vline(x=media_geral_cartoes, line_dash='dash', line_color='gray',
              annotation_text=f'M\u00e9dia: {media_geral_cartoes:.1f}', row=1, col=2)

fig.update_layout(
    title='Efeito Cl\u00e1ssico: Gols e Cart\u00f5es nos D\u00e9rbis<br><sup>Linha tracejada = m\u00e9dia geral do campeonato</sup>',
    template='plotly_white',
    width=1100,
    height=500,
    showlegend=False
)

fig.show()
_path = '../charts/classicos.html'
fig.write_html(_path, include_plotlyjs='cdn')

# Adiciona descrição estatística
_desc = "Comparação das médias de gols e cartões nos principais dérbis, com linha de referência na média geral do campeonato."
with open(_path) as f:
    html = f.read()
html = html.replace('<div>', f'<p style="text-align:center;color:#666;font-size:13px;font-family:sans-serif;margin:8px 0 0 0;">{_desc}</p>\n<div>', 1)
with open(_path, 'w') as f:
    f.write(html)

print('Gr\u00e1fico exportado: charts/classicos.html')

Gráfico exportado: charts/classicos.html


In [4]:
# PRESSÃO EM CASA: % vitórias mandante vs visitante por clube
GRANDES = ['Flamengo', 'Palmeiras', 'Corinthians', 'S\u00e3o Paulo', 'Santos',
           'Gr\u00eamio', 'Internacional', 'Cruzeiro', 'Atl\u00e9tico-MG',
           'Fluminense', 'Vasco', 'Botafogo']

results = []
for time in GRANDES:
    # Como mandante
    m = df[df['mandante'] == time]
    m_win = (m['mandante_Placar'] > m['visitante_Placar']).mean() * 100
    
    # Como visitante
    v = df[df['visitante'] == time]
    v_win = (v['visitante_Placar'] > v['mandante_Placar']).mean() * 100
    
    results.append({'time': time, 'vit_casa': m_win, 'vit_fora': v_win,
                    'diff': m_win - v_win, 'jogos_casa': len(m), 'jogos_fora': len(v)})

df_mandante = pd.DataFrame(results).sort_values('diff', ascending=True)

fig2 = go.Figure()

fig2.add_trace(go.Bar(
    y=df_mandante['time'],
    x=df_mandante['vit_casa'],
    name='Vit\u00f3rias em Casa',
    orientation='h',
    marker_color='#27ae60',
    text=df_mandante['vit_casa'].round(1).astype(str) + '%',
    textposition='inside'
))

fig2.add_trace(go.Bar(
    y=df_mandante['time'],
    x=df_mandante['vit_fora'],
    name='Vit\u00f3rias Fora',
    orientation='h',
    marker_color='#3498db',
    text=df_mandante['vit_fora'].round(1).astype(str) + '%',
    textposition='inside'
))

fig2.update_layout(
    title='Vit\u00f3rias em Casa vs Fora \u2014 Grandes Clubes<br><sup>Brasileir\u00e3o (2003-presente)</sup>',
    xaxis_title='% de Vit\u00f3rias',
    barmode='group',
    template='plotly_white',
    width=900,
    height=550,
    legend=dict(x=0.7, y=0.05)
)

fig2.show()
_path = '../charts/mandante_por_clube.html'
fig2.write_html(_path, include_plotlyjs='cdn')

# Adiciona descrição estatística
_desc = "Comparação pareada do aproveitamento em casa vs. fora por clube: todos os grandes clubes apresentam diferença estatística relevante entre as taxas de vitória como mandante e visitante."
with open(_path) as f:
    html = f.read()
html = html.replace('<div>', f'<p style="text-align:center;color:#666;font-size:13px;font-family:sans-serif;margin:8px 0 0 0;">{_desc}</p>\n<div>', 1)
with open(_path, 'w') as f:
    f.write(html)

print('Gr\u00e1fico exportado: charts/mandante_por_clube.html')

Gráfico exportado: charts/mandante_por_clube.html


In [5]:
# EFEITO NOVO TÉCNICO
# Identificar trocas de técnico: quando muda o técnico de um time entre jogos
df_sorted = df.sort_values(['ano', 'rodata'])

# Construir histórico de jogos por time (mandante ou visitante)
jogos_time = []
for _, row in df_sorted.iterrows():
    jogos_time.append({'time': row['mandante'], 'tecnico': row['tecnico_mandante'],
                       'ano': row['ano'], 'rodata': row['rodata'],
                       'gm': row['mandante_Placar'], 'gs': row['visitante_Placar'], 'ID': row['ID']})
    jogos_time.append({'time': row['visitante'], 'tecnico': row['tecnico_visitante'],
                       'ano': row['ano'], 'rodata': row['rodata'],
                       'gm': row['visitante_Placar'], 'gs': row['mandante_Placar'], 'ID': row['ID']})

df_jogos = pd.DataFrame(jogos_time)
df_jogos = df_jogos.dropna(subset=['tecnico'])
df_jogos = df_jogos.sort_values(['time', 'ano', 'rodata'])

# Detectar trocas
df_jogos['tecnico_ant'] = df_jogos.groupby(['time', 'ano'])['tecnico'].shift(1)
df_jogos['troca'] = (df_jogos['tecnico'] != df_jogos['tecnico_ant']) & df_jogos['tecnico_ant'].notna()

trocas = df_jogos[df_jogos['troca']].copy()
print(f'Total de trocas de t\u00e9cnico detectadas: {len(trocas)}')
print(f'M\u00e9dia de trocas por temporada: {len(trocas) / df["ano"].nunique():.1f}')

Total de trocas de técnico detectadas: 971
Média de trocas por temporada: 40.5


In [6]:
# Calcular aproveitamento 5 jogos antes e 5 jogos depois da troca
def calc_aproveitamento(pts_list):
    if not pts_list:
        return None
    return sum(pts_list) / (len(pts_list) * 3) * 100

resultados_troca = []

for _, troca_row in trocas.iterrows():
    time = troca_row['time']
    ano = troca_row['ano']
    rodada_troca = troca_row['rodata']
    
    jogos_time_ano = df_jogos[(df_jogos['time'] == time) & (df_jogos['ano'] == ano)].sort_values('rodata')
    jogos_time_ano = jogos_time_ano.reset_index(drop=True)
    
    idx_troca = jogos_time_ano[jogos_time_ano['rodata'] == rodada_troca].index
    if len(idx_troca) == 0:
        continue
    idx = idx_troca[0]
    
    # 5 jogos antes
    antes = jogos_time_ano.iloc[max(0, idx-5):idx]
    # 5 jogos depois (incluindo o jogo da troca)
    depois = jogos_time_ano.iloc[idx:idx+5]
    
    if len(antes) < 3 or len(depois) < 3:
        continue
    
    def pontos(row):
        if pd.isna(row['gm']) or pd.isna(row['gs']): return None
        if row['gm'] > row['gs']: return 3
        elif row['gm'] == row['gs']: return 1
        else: return 0
    
    pts_antes = [pontos(r) for _, r in antes.iterrows() if pontos(r) is not None]
    pts_depois = [pontos(r) for _, r in depois.iterrows() if pontos(r) is not None]
    
    aprov_antes = calc_aproveitamento(pts_antes)
    aprov_depois = calc_aproveitamento(pts_depois)
    
    if aprov_antes is not None and aprov_depois is not None:
        resultados_troca.append({
            'time': time, 'ano': ano, 'rodada': rodada_troca,
            'aprov_antes': aprov_antes, 'aprov_depois': aprov_depois,
            'diff': aprov_depois - aprov_antes
        })

df_troca = pd.DataFrame(resultados_troca)
print(f'Trocas analis\u00e1veis: {len(df_troca)}')
print(f'\nM\u00e9dia de aproveitamento:')
print(f'  Antes da troca: {df_troca["aprov_antes"].mean():.1f}%')
print(f'  Depois da troca: {df_troca["aprov_depois"].mean():.1f}%')
print(f'  Diferen\u00e7a m\u00e9dia: {df_troca["diff"].mean():+.1f} p.p.')
print(f'\nMelhorou: {(df_troca["diff"] > 0).sum()} ({(df_troca["diff"] > 0).mean()*100:.0f}%)')
print(f'Piorou: {(df_troca["diff"] < 0).sum()} ({(df_troca["diff"] < 0).mean()*100:.0f}%)')
print(f'Igual: {(df_troca["diff"] == 0).sum()}')

Trocas analisáveis: 861

Média de aproveitamento:
  Antes da troca: 39.5%
  Depois da troca: 44.3%
  Diferença média: +4.7 p.p.

Melhorou: 443 (51%)
Piorou: 333 (39%)
Igual: 85


In [7]:
# Histograma da diferença de aproveitamento pós-troca
fig3 = px.histogram(df_troca, x='diff', nbins=30,
                    title='Efeito Novo T\u00e9cnico: Mudan\u00e7a no Aproveitamento<br><sup>5 jogos antes vs 5 jogos depois da troca</sup>',
                    labels={'diff': 'Diferen\u00e7a de Aproveitamento (p.p.)', 'count': 'Frequ\u00eancia'},
                    color_discrete_sequence=['#3498db'])

fig3.add_vline(x=0, line_dash='dash', line_color='red', line_width=2)
fig3.add_vline(x=df_troca['diff'].mean(), line_dash='solid', line_color='green',
               annotation_text=f"M\u00e9dia: {df_troca['diff'].mean():+.1f} p.p.",
               annotation_position='top')

fig3.update_layout(template='plotly_white', width=800, height=450)

fig3.show()
_path = '../charts/efeito_tecnico.html'
fig3.write_html(_path, include_plotlyjs='cdn')

# Adiciona descrição estatística
_desc = "Distribuição da diferença de aproveitamento antes e após troca de técnico, com média de +4.7 p.p. A distribuição aproximadamente normal centrada levemente à direita indica um efeito positivo moderado."
with open(_path) as f:
    html = f.read()
html = html.replace('<div>', f'<p style="text-align:center;color:#666;font-size:13px;font-family:sans-serif;margin:8px 0 0 0;">{_desc}</p>\n<div>', 1)
with open(_path, 'w') as f:
    f.write(html)

print('Gr\u00e1fico exportado: charts/efeito_tecnico.html')

Gráfico exportado: charts/efeito_tecnico.html
